# Шаг 0.1
Скрипт на Python эмулирует поисковые запросы на страницу https://www.plasticsoldierreview.com/NewSearch.aspx сайта с обзорами наборов военно-исторических миниатюр. Эмуляция осуществляется для каждого года в заданном диапазоне от 2024 г. до н.э. до 2024 г. Скрипт собирает данные по всем обзорам за каждый год и для каждого набора определяет начало и конец временного периода на основе годов, в которых он был найден. Результат сохраняется в CSV с полями ID, Header, Years_from, Years_to.

## Как работает скрипт:
1. Перебор лет – от -2024 до 2024.
2. Поиск за год – для каждого года отправляется POST-запрос с одинаковым началом и концом.
3. Сбор данных – из результатов извлекаются ID и Header (название набора).
3. Накопление – для каждого ID запоминается текущий год.
4. Определение границ – после обработки всех лет для каждого ID вычисляются минимальный и максимальный год, в которых он был найден.
5. Сохранение – запись в CSV с нужными полями.

In [2]:
import requests
from bs4 import BeautifulSoup
import re
import csv
import time
from collections import defaultdict

def get_reviews_for_year(session, year):
    """
    Выполняет поиск по одному году и возвращает список словарей с ID и Header.
    """
    url = "https://www.plasticsoldierreview.com/NewSearch.aspx"
    
    # Загружаем страницу для получения свежих ViewState и EventValidation
    resp = session.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    viewstate = soup.find('input', {'name': '__VIEWSTATE'})['value']
    eventvalidation_tag = soup.find('input', {'name': '__EVENTVALIDATION'})
    eventvalidation = eventvalidation_tag['value'] if eventvalidation_tag else ''
    
    # Данные POST-запроса
    data = {
        '__VIEWSTATE': viewstate,
        '__EVENTVALIDATION': eventvalidation,
        '__LASTFOCUS': '',
        '__EVENTTARGET': '',
        '__EVENTARGUMENT': '',
        'ctl00$ContentPlaceHolder3$StartYearTextBox': str(year),
        'ctl00$ContentPlaceHolder3$EndYearTextBox': str(year),
        'ctl00$ContentPlaceHolder3$ManufacturerDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$PlasticColourDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$NationalityDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$TypeDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$MinimumScoreTextBox': '',
        'ctl00$ContentPlaceHolder3$AccuracyTextBox': '',
        'ctl00$ContentPlaceHolder3$PoseTextBox': '',
        'ctl00$ContentPlaceHolder3$PoseQtyTextBox': '',
        'ctl00$ContentPlaceHolder3$SculptingTextBox': '',
        'ctl00$ContentPlaceHolder3$MouldTextBox': '',
        'ctl00$ContentPlaceHolder3$SearchButton': 'Search',
    }
    
    response = session.post(url, data=data)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Ищем все ссылки на обзоры
    links = soup.find_all('a', href=re.compile(r'Review\.aspx\?id=\d+'))
    
    results = []
    for link in links:
        id_match = re.search(r'id=(\d+)', link['href'])
        if id_match:
            review_id = int(id_match.group(1))
            header = link.get_text(strip=True)
            results.append({'id': review_id, 'header': header})
    
    return results

def main():
    start_year = -2024
    end_year = 2024
    
    # Словарь: ключ - ID, значение - список годов
    years_by_id = defaultdict(list)
    # Словарь: ключ - ID, значение - Header (берём из первого появления)
    header_by_id = {}
    
    session = requests.Session()
    
    for year in range(start_year, end_year + 1):
        print(f"Обработка года: {year}")
        try:
            reviews = get_reviews_for_year(session, year)
            print(f"  Найдено обзоров: {len(reviews)}")
            for item in reviews:
                rid = item['id']
                years_by_id[rid].append(year)
                if rid not in header_by_id:
                    header_by_id[rid] = item['header']
        except Exception as e:
            print(f"  Ошибка при обработке года {year}: {e}")
        
        # Пауза между запросами – 1 секунда
        time.sleep(1)
    
    # Формируем список для CSV
    output = []
    for rid, years in years_by_id.items():
        years_sorted = sorted(years)
        years_from = min(years_sorted)
        years_to = max(years_sorted)
        header = header_by_id.get(rid, '')
        output.append({
            'ID': rid,
            'Header': header,
            'Years_from': years_from,
            'Years_to': years_to
        })
    
    # Сортируем по ID
    output.sort(key=lambda x: x['ID'])
    
    # Сохраняем в CSV
    with open('review_periods.csv', 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['ID', 'Header', 'Years_from', 'Years_to']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(output)
    
    print(f"Готово! Сохранено {len(output)} записей в review_periods.csv")

if __name__ == '__main__':
    main()

Обработка года: -2024
  Найдено обзоров: 28
Обработка года: -2023
  Найдено обзоров: 28
Обработка года: -2022
  Найдено обзоров: 28
Обработка года: -2021
  Найдено обзоров: 28
Обработка года: -2020
  Найдено обзоров: 28
Обработка года: -2019
  Найдено обзоров: 28
Обработка года: -2018
  Найдено обзоров: 28
Обработка года: -2017
  Найдено обзоров: 28
Обработка года: -2016
  Найдено обзоров: 28
Обработка года: -2015
  Найдено обзоров: 28
Обработка года: -2014
  Найдено обзоров: 28
Обработка года: -2013
  Найдено обзоров: 28
Обработка года: -2012
  Найдено обзоров: 28
Обработка года: -2011
  Найдено обзоров: 28
Обработка года: -2010
  Найдено обзоров: 28
Обработка года: -2009
  Найдено обзоров: 28
Обработка года: -2008
  Найдено обзоров: 28
Обработка года: -2007
  Найдено обзоров: 28
Обработка года: -2006
  Найдено обзоров: 28
Обработка года: -2005
  Найдено обзоров: 28
Обработка года: -2004
  Найдено обзоров: 28
Обработка года: -2003
  Найдено обзоров: 28
Обработка года: -2002
  Найдено 

# Шаг 0.2
Скрипт эмулирует поисковые запросы по полю Nationality, собирает для каждого обзора все национальности, в которых он был найден, и записывает их в одну строку с полями Nationality, Nationality_2, Nationality_3 и т.д. Максимальное количество столбцов определяется автоматически на основе данных.

## Как работает скрипт
1. Загрузка списка национальностей – загружается страница NewSearch.aspx, из выпадающего списка Nationality извлекаются все значения (кроме All). Для каждого значения сохраняется его числовой код и текстовое название.
2. Перебор национальностей – для каждой национальности выполняется POST-запрос с выбором этой национальности в поле NationalityDropdownlist. Остальные поля (даты, производитель, цвет и т.д.) остаются пустыми или равными 0 (все).
3. Сбор результатов – из страницы с результатами извлекаются все ссылки на обзоры (Review.aspx?id=...), из них берутся ID и Header (текст ссылки).
4. Формирование записей – для каждого найденного обзора создаётся запись с текущей национальностью. Если один и тот же ID встречается для нескольких национальностей, он будет записан несколько раз – по одному разу для каждой национальности.
5. Сохранение в CSV – все записи сохраняются в файл reviews_by_nationality.csv с полями ID, Header, Nationality.

In [3]:
import requests
from bs4 import BeautifulSoup
import re
import csv
import time
from collections import defaultdict

def get_nationalities(session):
    """
    Загружает страницу NewSearch.aspx, извлекает все опции выпадающего списка Nationality,
    возвращает словарь {value: text} для всех значений, кроме 'All' (обычно value='0').
    """
    url = "https://www.plasticsoldierreview.com/NewSearch.aspx"
    resp = session.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    select = soup.find('select', {'name': 'ctl00$ContentPlaceHolder3$NationalityDropdownlist'})
    if not select:
        select = soup.find('select', {'id': 'ctl00_ContentPlaceHolder3_NationalityDropdownlist'})
    
    nationalities = {}
    if select:
        for option in select.find_all('option'):
            value = option.get('value')
            text = option.get_text(strip=True)
            if value != '0':  # пропускаем 'All'
                nationalities[value] = text
    return nationalities

def search_by_nationality(session, nat_value, nat_text):
    """
    Выполняет поиск по заданной национальности (nat_value) и возвращает список словарей
    с ключами 'id' и 'header' для всех найденных обзоров.
    """
    url = "https://www.plasticsoldierreview.com/NewSearch.aspx"
    
    resp = session.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    viewstate = soup.find('input', {'name': '__VIEWSTATE'})['value']
    eventvalidation_tag = soup.find('input', {'name': '__EVENTVALIDATION'})
    eventvalidation = eventvalidation_tag['value'] if eventvalidation_tag else ''
    
    data = {
        '__VIEWSTATE': viewstate,
        '__EVENTVALIDATION': eventvalidation,
        '__LASTFOCUS': '',
        '__EVENTTARGET': '',
        '__EVENTARGUMENT': '',
        'ctl00$ContentPlaceHolder3$StartYearTextBox': '',
        'ctl00$ContentPlaceHolder3$EndYearTextBox': '',
        'ctl00$ContentPlaceHolder3$ManufacturerDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$PlasticColourDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$NationalityDropdownlist': nat_value,
        'ctl00$ContentPlaceHolder3$TypeDropdownlist': '0',
        'ctl00$ContentPlaceHolder3$MinimumScoreTextBox': '',
        'ctl00$ContentPlaceHolder3$AccuracyTextBox': '',
        'ctl00$ContentPlaceHolder3$PoseTextBox': '',
        'ctl00$ContentPlaceHolder3$PoseQtyTextBox': '',
        'ctl00$ContentPlaceHolder3$SculptingTextBox': '',
        'ctl00$ContentPlaceHolder3$MouldTextBox': '',
        'ctl00$ContentPlaceHolder3$SearchButton': 'Search',
    }
    
    response = session.post(url, data=data)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    links = soup.find_all('a', href=re.compile(r'Review\.aspx\?id=\d+'))
    
    results = []
    for link in links:
        id_match = re.search(r'id=(\d+)', link['href'])
        if id_match:
            review_id = int(id_match.group(1))
            header = link.get_text(strip=True)
            results.append({'id': review_id, 'header': header})
    
    return results

def main():
    session = requests.Session()
    
    print("Загрузка списка национальностей...")
    nationalities = get_nationalities(session)
    print(f"Найдено национальностей: {len(nationalities)}")
    
    # Словарь: ID -> set of nationality names
    id_to_nationalities = defaultdict(set)
    # Словарь: ID -> header (берём из первого найденного обзора, они одинаковые)
    id_to_header = {}
    
    for value, text in nationalities.items():
        print(f"Обработка национальности: {text} (value={value})")
        try:
            reviews = search_by_nationality(session, value, text)
            print(f"  Найдено обзоров: {len(reviews)}")
            for item in reviews:
                rid = item['id']
                id_to_nationalities[rid].add(text)
                if rid not in id_to_header:
                    id_to_header[rid] = item['header']
        except Exception as e:
            print(f"  Ошибка при обработке национальности {text}: {e}")
        
        time.sleep(1)
    
    # Определяем максимальное количество национальностей у одного ID
    max_count = max((len(nats) for nats in id_to_nationalities.values()), default=0)
    print(f"Максимальное количество национальностей у одного набора: {max_count}")
    
    # Формируем список записей для CSV
    output_rows = []
    for rid, nats_set in id_to_nationalities.items():
        # Сортируем национальности для детерминированного порядка
        sorted_nats = sorted(nats_set)
        row = {'ID': rid, 'Header': id_to_header.get(rid, '')}
        # Заполняем столбцы Nationality_1, Nationality_2, ...
        for i, nat in enumerate(sorted_nats, start=1):
            row[f'Nationality_{i}'] = nat
        # Остальные столбцы оставляем пустыми (будут заполнены пустыми строками при записи)
        output_rows.append(row)
    
    # Сортируем по ID
    output_rows.sort(key=lambda x: x['ID'])
    
    # Определяем имена полей: ID, Header, Nationality_1, Nationality_2, ... до max_count
    fieldnames = ['ID', 'Header'] + [f'Nationality_{i}' for i in range(1, max_count + 1)]
    
    # Записываем CSV
    with open('reviews_by_nationality_wide.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, restval='')
        writer.writeheader()
        writer.writerows(output_rows)
    
    print(f"Готово! Сохранено {len(output_rows)} записей в reviews_by_nationality_wide.csv")

if __name__ == '__main__':
    main()

Загрузка списка национальностей...
Найдено национальностей: 183
Обработка национальности: Afghan (value=1)
  Найдено обзоров: 16
Обработка национальности: Akkadian (value=178)
  Найдено обзоров: 0
Обработка национальности: Albanian (value=118)
  Найдено обзоров: 14
Обработка национальности: Algerian (value=2)
  Найдено обзоров: 18
Обработка национальности: Almoravid (value=141)
  Найдено обзоров: 6
Обработка национальности: Anatolian (value=3)
  Найдено обзоров: 6
Обработка национальности: Andalusian (value=150)
  Найдено обзоров: 8
Обработка национальности: Anglo-Saxon (value=4)
  Найдено обзоров: 16
Обработка национальности: Arab (value=5)
  Найдено обзоров: 28
Обработка национальности: Aramaic (value=170)
  Найдено обзоров: 2
Обработка национальности: Argentinian (value=6)
  Найдено обзоров: 0
Обработка национальности: Armenian (value=169)
  Найдено обзоров: 2
Обработка национальности: Assyrian (value=7)
  Найдено обзоров: 24
Обработка национальности: Australian (value=8)
  Найдено 

# Шаг 0.3
Скрипт обогащает данные о наборах миниатюр пластиковых исторических миниатюрах путём парсинга данных с сайта https://www.plasticsoldierreview.com. В результате будут получены дополнительные характеристики наборов: год выпуска, количество фигур, совокупный рейтинг качества. Эти данные будут использованы для последующего анализа.
Результат: файл psr_enriched_data.csv с полями: ID, Header, Release_Year, Aggregate_Rating, Num_Figures

## Как работает скрипт
1. Обработка обзоров: Если набор не имеет собственных данных, скрипт ищет ссылку на оригинальный обзор и использует его данные.
2. Суммирование количества фигур: Корректно обрабатывает сложные форматы.
3. Защита от рекурсии: Параметр allow_alt_search=False при рекурсивном вызове предотвращает бесконечные циклы.
4. Типы данных: Использование Int64 позволяет хранить целые числа с пропусками без преобразования в float.
5. Промежуточное сохранение: Каждые 100 записей результат сохраняется во временный файл для защиты от потери данных при сетевых сбоях.

In [1]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import re
import time
from tqdm import tqdm

# --- 1. Загрузка текущего файла и исправление типов данных ---
print("Загрузка файла...")
df = pd.read_csv('psr_enriched_data.csv')

# Переводим Release_Year и Num_Figures в Int64 (целые числа с поддержкой пропусков)
df['Release_Year'] = df['Release_Year'].astype('Int64')
df['Num_Figures'] = df['Num_Figures'].astype('Int64')

print("Типы данных исправлены.")
print(df[['Release_Year', 'Num_Figures']].head())

# --- 2. Настройка стабильной сессии для дозабора рейтингов ---
session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5, status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

def get_aggregate_rating(review_id):
    """
    Собирает ТОЛЬКО Aggregate_Rating для конкретного ID.
    Возвращает сумму 5 оценок или None.
    """
    url = f"https://www.plasticsoldierreview.com/Review.aspx?id={review_id}"
    try:
        response = session.get(url, timeout=15)
        if response.status_code != 200:
            return None
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Ищем заголовок "Ratings (out of 10)"
        ratings_header = soup.find(string=re.compile(r'Ratings\s*\(out of 10\)', re.I))
        if not ratings_header:
            return None
            
        # Ищем список (ul или ol), который идет сразу после заголовка
        parent = ratings_header.find_parent()
        next_list = parent.find_next(['ul', 'ol'])
        
        if not next_list:
            return None
            
        # Извлекаем последнее число из каждого элемента списка
        items = next_list.find_all('li')
        ratings = []
        for item in items:
            # Ищем числа от 1 до 10
            numbers = re.findall(r'\b(10|[1-9])\b', item.get_text())
            if numbers:
                ratings.append(int(numbers[-1]))
                
        # Проверяем, что собрали ровно 5 оценок
        if len(ratings) == 5:
            return sum(ratings)
        return None
        
    except Exception as e:
        return None

# --- 3. Дозабор рейтингов ---
# Находим индексы строк, где рейтинг отсутствует
missing_ratings_idx = df[df['Aggregate_Rating'].isna()].index
print(f"\nРейтинги отсутствуют в {len(missing_ratings_idx)} строках. Начинаем дозабор...")

# Идем по пропущенным строкам с прогресс-баром
for idx in tqdm(missing_ratings_idx, desc="Сбор рейтингов"):
    review_id = df.loc[idx, 'ID']
    rating = get_aggregate_rating(review_id)
    
    if rating is not None:
        # Записываем рейтинг как целое число
        df.at[idx, 'Aggregate_Rating'] = rating
        
    # Задержка, чтобы не получить бан
    time.sleep(0.5)
    
    # Промежуточное сохранение каждые 100 записей
    # (на случай, если интернет отвалится на середине)
    if len(missing_ratings_idx) % 100 == 0:
        df.to_csv('psr_enriched_data_temp.csv', index=False, encoding='utf-8')

# --- 4. Финальное сохранение ---
# Явно приводим колонку рейтинга к Int64 перед сохранением
df['Aggregate_Rating'] = df['Aggregate_Rating'].astype('Int64')

df.to_csv('psr_enriched_data.csv', index=False, encoding='utf-8')

print("\n=== Готово! ===")
print(f"Файл 'psr_enriched_data.csv' обновлен.")
print(f"Найдено рейтингов: {df['Aggregate_Rating'].notna().sum()} из {len(df)}")

print("\nПример обновленных данных:")
print(df[['ID', 'Header', 'Release_Year', 'Aggregate_Rating', 'Num_Figures']].head(10))

Загрузка файла...
Типы данных исправлены.
   Release_Year  Num_Figures
0          <NA>           50
1          <NA>           50
2          1991           50
3          <NA>           18
4          <NA>           31

Рейтинги отсутствуют в 2837 строках. Начинаем дозабор...


Сбор рейтингов: 100%|██████████████████████████████████████████████████████████████| 2837/2837 [34:41<00:00,  1.36it/s]


=== Готово! ===
Файл 'psr_enriched_data.csv' обновлен.
Найдено рейтингов: 2082 из 2837

Пример обновленных данных:
   ID                                             Header  Release_Year  \
0   1                   Accurate British Infantry (7200)          <NA>   
1   2                   Accurate American Militia (7201)          <NA>   
2   3                     Accurate Union Infantry (7202)          1991   
3   4               Accurate Union Artillery Team (7204)          <NA>   
4   5                     Accurate Union Pioneers (7205)          <NA>   
5   6  Accurate Hundred Years War English Men-At-Arms...          <NA>   
6   7  Accurate Hundred Years War Knights Of France (...          <NA>   
7   8         Accurate Confederate Artillery Team (7208)          <NA>   
8   9               Accurate Confederate Pioneers (7209)          <NA>   
9  10            Accurate Waterloo French Cavalry (7212)          1969   

   Aggregate_Rating  Num_Figures  
0                47           50  